In [ ]:
!pip install bert-score

In [2]:
file_path = "/content/fact_check_sample.json"

In [4]:
import json
import torch
from bert_score import score
from tqdm import tqdm
from collections import defaultdict

In [5]:
model_type = "microsoft/deberta-xlarge-mnli"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [6]:
summary_scores = []
model_scores = defaultdict(list)

In [ ]:
with open(file_path, "r") as f:
    for line in tqdm(f, desc="Processing data"):
        data = json.loads(line.strip())

        reference = [data["reference"]]
        sentences = [" ".join(data["sentences"])]
        model = data["model"]

        # Calculate BERT Scores
        P, R, F1 = score(sentences, reference, lang="en", model_type=model_type, device=device)

        summary_scores.append({
            "doc_id": data["doc_id"],
            "model": model,
            "BERTScore_Precision": P.item(),
            "BERTScore_Recall": R.item(),
            "BERTScore_F1": F1.item()
        })

        model_scores[model].append({
            "BERTScore_Precision": P.item(),
            "BERTScore_Recall": R.item(),
            "BERTScore_F1": F1.item()
        })

In [ ]:
model_avg_scores = {}
for model, scores in model_scores.items():
    P_avg = sum(score['BERTScore_Precision'] for score in scores) / len(scores)
    R_avg = sum(score['BERTScore_Recall'] for score in scores) / len(scores)
    F1_avg = sum(score['BERTScore_F1'] for score in scores) / len(scores)
    model_avg_scores[model] = {
        "BERTScore_Precision_Avg": P_avg,
        "BERTScore_Recall_Avg": R_avg,
        "BERTScore_F1_Avg": F1_avg
    }

print("Individual Summary Scores:")
for score in summary_scores:
    print(f"Doc ID: {score['doc_id']}, Model: {score['model']}, BERTScore-P: {score['BERTScore_Precision']:.4f}, BERTScore-R: {score['BERTScore_Recall']:.4f}, BERTScore-F1: {score['BERTScore_F1']:.4f}")

print("\nModel-Level Average Scores:")
for model, avg_score in model_avg_scores.items():
    print(f"Model: {model}, BERTScore-P: {avg_score['BERTScore_Precision_Avg']:.4f}, BERTScore-R: {avg_score['BERTScore_Recall_Avg']:.4f}, BERTScore-F1: {avg_score['BERTScore_F1_Avg']:.4f}")
